# Wan 2.1 — text-to-video test (1.3B on free Colab T4)

Generates a short video from a text prompt with **Wan 2.1 T2V 1.3B**
(`Wan-AI/Wan2.1-T2V-1.3B`, Apache 2.0) via `diffusers` on a free Colab **T4**.
Output: ~2 s, 832×480, 33 frames — a proof that the pipeline runs, not a
production shot. The 14B model needs 16 GB+ VRAM and does NOT fit a free T4.

**Status: UNTESTED** — model repo + license verified via the HF API
(Apache 2.0 in the model card), but no Colab run has happened yet.
Note: the official demo Space `Wan-AI/Wan2.1` is currently **RUNTIME_ERROR**
(down as of 2026-11-08), so Colab is the working path. Report the failing cell.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory // (1024**2), 'MiB')

In [ ]:
print('Install diffusers stack...')
!pip install -q diffusers transformers accelerate imageio imageio-ffmpeg
!python -c "import diffusers; print('diffusers', diffusers.__version__)"

In [ ]:
import torch
from diffusers import WanPipeline

print('Load Wan2.1-T2V-1.3B (first run downloads ~2.5 GB)...')
pipe = WanPipeline.from_pretrained('Wan-AI/Wan2.1-T2V-1.3B', torch_dtype=torch.bfloat16)
pipe.to('cuda')
print('model loaded')

In [ ]:
prompt = 'A quiet morning in the Casbah of Algiers, golden sunlight over whitewashed houses and the sea, cinematic, slow camera push-in'
print('Generate 33 frames (832x480)...')
out = pipe(prompt=prompt, num_frames=33, height=480, width=832,
           num_inference_steps=25, guidance_scale=5.0)
frames = out.frames[0]
print('frames:', len(frames), 'size:', frames[0].size)

In [ ]:
import imageio
with imageio.get_writer('wan21_test.mp4', fps=16) as w:
    for f in frames:
        w.append_data(f)
print('saved wan21_test.mp4')
!ffprobe -v error -show_entries stream=width,height,duration,r_frame_rate -of csv wan21_test.mp4

## Troubleshooting

- **CUDA out of memory:** add `pipe.enable_model_cpu_offload()` after loading,
  or drop to 16 frames / 480×480. The 1.3B model needs ~8 GB; T4 has 15 GB.
- **Slow generation:** 25 steps × 33 frames takes several minutes on T4 — normal.
- **Image-to-video (next step):** use `WanImageToVideoPipeline` with
  `Wan-AI/Wan2.1-I2V-14B-480P` — 14B, needs more VRAM; try 720P only on Kaggle P100.
- **Wan 2.2:** current generation (`Wan-AI/Wan2.2-T2V-A14B`) is heavier; 2.1 1.3B
  stays the free-T4 choice. Re-test 2.2 only if a bigger free GPU is available.
- **Use in production:** Wan clips are AI-generated inserts (establishing shots,
  backgrounds) — never the hero footage; Blender/Manim stay the main engine.